In [4]:
import torch
import json
import time
import os
import numpy as np
import pandas as pd
from transformers import BertTokenizer, BertForSequenceClassification
from openai import OpenAI
from dotenv import load_dotenv

# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ── Config ────────────────────────────────────────────────────────────────
AGENT_MODEL         = "gpt-4o"       # smarter model for clarification
AMBIGUITY_THRESHOLD = 0.65           # confidence below this → Ambiguous
MAX_ITERATIONS      = 3              # max clarification rounds per input

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"Agent  : {AGENT_MODEL}")
print(f"Threshold : {AMBIGUITY_THRESHOLD}")

Device : cuda
Agent  : gpt-4o
Threshold : 0.65


In [5]:
# Load trained BERT model for requirement classification
MODEL_PATH = '../models/bert_requirements_final'

tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model     = BertForSequenceClassification.from_pretrained(MODEL_PATH)
model     = model.to(device)
model.eval()

# Load label maps
with open(f'{MODEL_PATH}/label2id.json') as f:
    label2id = json.load(f)
with open(f'{MODEL_PATH}/id2label.json') as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

print(f"Model loaded from {MODEL_PATH}")
print(f"Labels: {list(label2id.keys())}")

Model loaded from ../models/bert_requirements_final
Labels: ['Ambiguous', 'FR', 'NFR_Legal', 'NFR_LookAndFeel', 'NFR_Maintainability', 'NFR_Operational', 'NFR_Other', 'NFR_Performance', 'NFR_Portability', 'NFR_Reliability', 'NFR_Scalability', 'NFR_Security', 'NFR_Usability']


In [7]:
# ── Classification Function ───────────────────────────────────────────────
def classify_requirement(text, threshold=AMBIGUITY_THRESHOLD):
    """
    Classify a requirement statement using fine-tuned BERT.
    Returns label, confidence score, and all class probabilities.
    If confidence < threshold → returns 'Ambiguous'.
    """
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding='max_length',
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs      = torch.softmax(outputs.logits, dim=1).squeeze()
    confidence = probs.max().item()
    pred_id    = probs.argmax().item()
    pred_label = id2label[pred_id]

    # All class probabilities for transparency
    all_probs = {
        id2label[i]: round(probs[i].item(), 4)
        for i in range(len(id2label))
    }

    # Override with Ambiguous if confidence is too low
    if confidence < threshold:
        return 'Ambiguous', confidence, all_probs

    return pred_label, confidence, all_probs

# ── Test ──────────────────────────────────────────────────────────────────
test_cases = [
    "The system shall encrypt all user passwords using AES-256.",
    "The system shall respond within 2 seconds under peak load.",
    "The system should be good.",         # vague → should be Ambiguous
    "It must work properly for users.",   # vague → should be Ambiguous
]

print("=== CLASSIFIER TEST ===\n")
for text in test_cases:
    label, conf, _ = classify_requirement(text)
    status = "AMBIGUOUS" if label == 'Ambiguous' else f" {label}"
    print(f"Input      : {text}")
    print(f"Result     : {status}")
    print(f"Confidence : {conf:.4f}\n")

=== CLASSIFIER TEST ===

Input      : The system shall encrypt all user passwords using AES-256.
Result     :  NFR_Security
Confidence : 0.9968

Input      : The system shall respond within 2 seconds under peak load.
Result     :  NFR_Performance
Confidence : 0.9624

Input      : The system should be good.
Result     : AMBIGUOUS
Confidence : 0.9990

Input      : It must work properly for users.
Result     : AMBIGUOUS
Confidence : 0.9990



In [8]:
# ── Ambiguity Analysis Function ─────────────────────────────────────────
def analyze_ambiguity(text):
    """
    Use GPT-4o to identify WHICH properties are missing or unclear.
    Returns a structured dict of missing properties and their explanations.
    """
    prompt = f"""You are a requirements engineering expert trained in IEEE 830.

Analyze this requirement statement for ambiguity:
"{text}"

Check each property and determine if it is CLEAR or UNCLEAR:
- Actor: Who or what performs the action? (system, user, admin?)
- Action: What should happen? Is the verb specific?
- Object: What is the action performed on?
- Condition: Under what circumstances or preconditions?
- Measurable criterion: Is there a quantifiable target? (for NFRs: numbers, units, thresholds)
- Scope: Is the boundary of the requirement clear?

Respond ONLY in this exact JSON format:
{{
  "missing_properties": ["property1", "property2"],
  "issues": {{
    "Actor": "clear OR brief explanation of issue",
    "Action": "clear OR brief explanation of issue",
    "Object": "clear OR brief explanation of issue",
    "Condition": "clear OR brief explanation of issue",
    "Measurable criterion": "clear OR brief explanation of issue",
    "Scope": "clear OR brief explanation of issue"
  }},
  "ambiguity_severity": "high/medium/low",
  "summary": "one sentence explaining the main issue"
}}"""

    try:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a requirements engineering analyzer. Respond only in valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=500,
            temperature=0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)

    except Exception as e:
        print(f"API error: {e}")
        return {
            "missing_properties": ["Action", "Measurable criterion"],
            "issues": {},
            "ambiguity_severity": "medium",
            "summary": "Could not analyze — defaulting to generic clarification"
        }

# ── Test ──────────────────────────────────────────────────────────────────
test_vague = "The system should be good."
analysis   = analyze_ambiguity(test_vague)

print("=== AMBIGUITY ANALYSIS ===\n")
print(f"Input    : {test_vague}")
print(f"Severity : {analysis['ambiguity_severity']}")
print(f"Summary  : {analysis['summary']}")
print(f"Missing  : {analysis['missing_properties']}")
print(f"\nDetailed issues:")
for prop, issue in analysis['issues'].items():
    print(f"  {prop:<25}: {issue}")

=== AMBIGUITY ANALYSIS ===

Input    : The system should be good.
Severity : high
Summary  : The requirement is vague and lacks specificity in all key areas, making it highly ambiguous.
Missing  : ['Object', 'Condition', 'Measurable criterion', 'Scope']

Detailed issues:
  Actor                    : Unclear; 'system' is mentioned but not specific enough.
  Action                   : Unclear; 'should be good' is vague and lacks specificity.
  Object                   : Unclear; no specific object is mentioned.
  Condition                : Unclear; no conditions or preconditions are specified.
  Measurable criterion     : Unclear; no quantifiable target or criteria are provided.
  Scope                    : Unclear; the boundary of the requirement is not defined.


In [9]:
# ── Clarification Question Generation Function ─────────────────────────
def generate_clarification_questions(text, analysis):
    """
    Generate targeted clarification questions based on ambiguity analysis.
    Questions are specific to what is missing — not generic.
    """
    missing    = analysis.get('missing_properties', [])
    issues     = analysis.get('issues', {})
    severity   = analysis.get('ambiguity_severity', 'medium')
    summary    = analysis.get('summary', '')

    # Limit questions: high severity → 3, medium → 2, low → 1
    max_questions = {'high': 3, 'medium': 2, 'low': 1}.get(severity, 2)

    prompt = f"""You are a requirements engineer conducting a stakeholder interview.

The following requirement statement is ambiguous:
"{text}"

Ambiguity analysis:
- Main issue: {summary}
- Missing properties: {', '.join(missing)}
- Detailed issues: {json.dumps(issues, indent=2)}

Generate exactly {max_questions} clarification question(s) to resolve the ambiguity.

Rules:
1. Each question must target a SPECIFIC missing property
2. Questions must be simple enough for a non-technical stakeholder
3. Do NOT ask multiple things in one question
4. Do NOT ask yes/no questions — ask open-ended questions
5. Do NOT repeat the requirement back to the user
6. Make questions conversational and friendly

Respond ONLY in this exact JSON format:
{{
  "questions": [
    {{
      "targets": "property name this question resolves",
      "question": "the actual question to ask the stakeholder"
    }}
  ]
}}"""

    try:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a requirements elicitation expert. Respond only in valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=400,
            temperature=0.2,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
        return result.get('questions', [])

    except Exception as e:
        print(f"API error: {e}")
        return [{"targets": "Action", "question": "Could you describe more specifically what the system should do?"}]

# ── Test ──────────────────────────────────────────────────────────────────
questions = generate_clarification_questions(test_vague, analysis)

print("=== GENERATED CLARIFICATION QUESTIONS ===\n")
for i, q in enumerate(questions, 1):
    print(f"Q{i} [{q['targets']}]: {q['question']}")

=== GENERATED CLARIFICATION QUESTIONS ===

Q1 [Actor]: Who specifically will be using the system, and what are their main goals?
Q2 [Measurable criterion]: What specific outcomes or results would indicate that the system is performing well?
Q3 [Scope]: What are the main boundaries or limitations we should consider for the system?


In [10]:
# ── Requirement Refinement Function ───────────────────────────────────
def refine_requirement(original_text, questions, answers):
    """
    Use GPT-4o to rewrite the original vague requirement into a
    clear, well-structured requirement using the stakeholder's answers.
    """
    qa_pairs = "\n".join([
        f"Q: {q['question']}\nA: {a}"
        for q, a in zip(questions, answers)
    ])

    prompt = f"""You are a requirements engineer.

Original vague requirement:
"{original_text}"

Clarification obtained from stakeholder:
{qa_pairs}

Rewrite this as a single, clear, well-structured software requirement that:
1. Follows the format: "The system shall [action] [object] [condition/constraint]"
2. Includes any measurable criteria from the stakeholder's answers
3. Is unambiguous and testable
4. Keeps the original intent intact

Respond ONLY in this exact JSON format:
{{
  "refined_requirement": "the rewritten requirement",
  "improvements": ["improvement 1", "improvement 2"]
}}"""

    try:
        response = client.chat.completions.create(
            model=AGENT_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a requirements engineering expert. Respond only in valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=300,
            temperature=0,
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)

    except Exception as e:
        print(f"API error: {e}")
        return {"refined_requirement": original_text, "improvements": []}

In [11]:
# ── Full Clarification Loop ─────────────────────────────────────────────
def run_clarification_loop(text, interactive=True):
    """
    Full agent loop:
    1. Classify input with BERT
    2. If ambiguous → analyze + generate questions
    3. Collect stakeholder answers
    4. Refine requirement
    5. Re-classify refined requirement
    6. Repeat up to MAX_ITERATIONS times

    interactive=True  → prompts user for input (notebook use)
    interactive=False → returns structure for testing/API use
    """
    print(f"\n{'='*60}")
    print(f"INPUT: {text}")
    print(f"{'='*60}")

    original_text = text
    iteration     = 0
    history       = []

    while iteration < MAX_ITERATIONS:
        iteration += 1
        print(f"\n── Iteration {iteration}/{MAX_ITERATIONS} ──")

        # Step 1: Classify
        label, confidence, all_probs = classify_requirement(text)
        print(f"Classification : {label}")
        print(f"Confidence     : {confidence:.4f}")

        # Step 2: If not ambiguous — done
        if label != 'Ambiguous':
            print(f"\nRequirement classified successfully!")
            print(f"Label      : {label}")
            print(f"Confidence : {confidence:.4f}")

            history.append({
                'iteration'   : iteration,
                'text'        : text,
                'label'       : label,
                'confidence'  : confidence,
                'status'      : 'classified'
            })
            return {
                'original_text'     : original_text,
                'final_text'        : text,
                'label'             : label,
                'confidence'        : confidence,
                'iterations'        : iteration,
                'history'           : history,
                'status'            : 'success'
            }

        # Step 3: Analyze ambiguity
        print(f"\nAmbiguous (confidence: {confidence:.4f}) — analyzing...")
        analysis = analyze_ambiguity(text)
        print(f"Severity : {analysis['ambiguity_severity']}")
        print(f"Issue    : {analysis['summary']}")

        # Step 4: Generate clarification questions
        questions = generate_clarification_questions(text, analysis)
        print(f"\nClarification questions:")
        for i, q in enumerate(questions, 1):
            print(f"Q{i}: {q['question']}")

        # Step 5: Collect answers
        answers = []
        if interactive:
            print()
            for i, q in enumerate(questions, 1):
                answer = input(f"   A{i}: ").strip()
                answers.append(answer if answer else "Not specified")
        else:
            # Non-interactive mode — use placeholder for testing
            answers = ["Not specified"] * len(questions)

        # Step 6: Refine requirement
        refinement = refine_requirement(text, questions, answers)
        refined    = refinement['refined_requirement']

        print(f"\nRefined requirement:")
        print(f"{refined}")

        history.append({
            'iteration'   : iteration,
            'original'    : text,
            'questions'   : questions,
            'answers'     : answers,
            'refined'     : refined,
            'label'       : label,
            'confidence'  : confidence,
            'status'      : 'clarified'
        })

        # Use refined text for next iteration
        text = refined

    # Max iterations reached — flag for manual review
    print(f"\nMax iterations ({MAX_ITERATIONS}) reached.")
    print(f"Flagging for manual review.")

    return {
        'original_text' : original_text,
        'final_text'    : text,
        'label'         : 'Requires Manual Review',
        'confidence'    : 0.0,
        'iterations'    : MAX_ITERATIONS,
        'history'       : history,
        'status'        : 'manual_review'
    }

In [13]:
# ── Test Cases for Full Clarification Loop ─────────────────────────────
# Test 1: Clear requirement — should classify immediately
print("TEST 1: Clear requirement")
result1 = run_clarification_loop(
    "The system shall encrypt all passwords using AES-256 encryption.",
    interactive=False
)

print(f"\nFinal result: {result1['label']} ({result1['confidence']:.4f})")
print(f"Iterations  : {result1['iterations']}")

# Test 2: Vague requirement — should trigger clarification loop
print("\n\nTEST 2: Vague requirement (interactive)")
result2 = run_clarification_loop(
    "The system should be easy to use.",
    interactive=True   # prompt answers
)

print(f"\nFinal result: {result2['label']} ({result2['confidence']:.4f})")
print(f"Iterations  : {result2['iterations']}")

TEST 1: Clear requirement

INPUT: The system shall encrypt all passwords using AES-256 encryption.

── Iteration 1/3 ──
Classification : NFR_Security
Confidence     : 0.9952

Requirement classified successfully!
Label      : NFR_Security
Confidence : 0.9952

Final result: NFR_Security (0.9952)
Iterations  : 1


TEST 2: Vague requirement (interactive)

INPUT: The system should be easy to use.

── Iteration 1/3 ──
Classification : Ambiguous
Confidence     : 0.9990

Ambiguous (confidence: 0.9990) — analyzing...
Severity : high
Issue    : The main issue is the lack of specificity and measurable criteria for 'easy to use'.

Clarification questions:
Q1: Who do you envision will be using the system most frequently?
Q2: What specific features or aspects would make the system feel easy to use for you?
Q3: Can you describe what tasks or processes you expect to be straightforward in the system?


Refined requirement:
The system shall allow users to complete the checkout process in under 3 clicks 

In [14]:
# ── Requirements Registry ─────────────────────────────────────────────
import sqlite3
from datetime import datetime

class RequirementsRegistry:
    """
    Persistent store for all classified requirements.
    Used by the SRS generator in Phase 4.
    """

    def __init__(self, db_path='../data/requirements_registry.db'):
        self.db_path = db_path
        self._init_db()

    def _init_db(self):
        conn = sqlite3.connect(self.db_path)
        conn.execute('''
            CREATE TABLE IF NOT EXISTS requirements (
                id              INTEGER PRIMARY KEY AUTOINCREMENT,
                original_text   TEXT NOT NULL,
                final_text      TEXT NOT NULL,
                label           TEXT NOT NULL,
                confidence      REAL,
                iterations      INTEGER,
                status          TEXT,
                timestamp       TEXT,
                project_name    TEXT
            )
        ''')
        conn.commit()
        conn.close()
        print(f"Registry initialized at {self.db_path}")

    def add(self, result, project_name="Default Project"):
        conn = sqlite3.connect(self.db_path)
        conn.execute('''
            INSERT INTO requirements
            (original_text, final_text, label, confidence,
             iterations, status, timestamp, project_name)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            result['original_text'],
            result['final_text'],
            result['label'],
            result['confidence'],
            result['iterations'],
            result['status'],
            datetime.now().isoformat(),
            project_name
        ))
        conn.commit()
        conn.close()

    def get_all(self, project_name=None):
        conn = sqlite3.connect(self.db_path)
        if project_name:
            rows = conn.execute(
                'SELECT * FROM requirements WHERE project_name = ?',
                (project_name,)
            ).fetchall()
        else:
            rows = conn.execute('SELECT * FROM requirements').fetchall()
        conn.close()

        cols = ['id','original_text','final_text','label',
                'confidence','iterations','status','timestamp','project_name']
        return pd.DataFrame(rows, columns=cols)

    def summary(self, project_name=None):
        df = self.get_all(project_name)
        if df.empty:
            print("Registry is empty.")
            return
        print(f"Total requirements : {len(df)}")
        print(f"\nBy label:")
        print(df['label'].value_counts().to_string())
        print(f"\nBy status:")
        print(df['status'].value_counts().to_string())

# Initialize registry
registry = RequirementsRegistry()

# Save test results
registry.add(result1, project_name="Demo Project")
if result2['status'] != 'manual_review':
    registry.add(result2, project_name="Demo Project")

registry.summary(project_name="Demo Project")

Registry initialized at ../data/requirements_registry.db
Total requirements : 2

By label:
label
NFR_Security     1
NFR_Usability    1

By status:
status
success    2


In [20]:
# ── Batch Processing Function ─────────────────────────────────────────
def process_requirements_batch(requirements_list, project_name="Default Project"):
    """
    Process a list of requirements non-interactively.
    Ambiguous ones are flagged for manual review.
    Useful for testing and evaluating the agent at scale.
    """
    results = []
    flagged = []

    print(f"Processing {len(requirements_list)} requirements...\n")

    for i, text in enumerate(requirements_list, 1):
        print(f"[{i}/{len(requirements_list)}] {text[:60]}...")

        label, confidence, _ = classify_requirement(text)

        if label == 'Ambiguous':
            # Non-interactive — flag instead of asking questions
            result = {
                'original_text' : text,
                'final_text'    : text,
                'label'         : 'Requires Manual Review',
                'confidence'    : confidence,
                'iterations'    : 0,
                'status'        : 'flagged_ambiguous'
            }
            flagged.append(text)
            print(f"Flagged as ambiguous (conf: {confidence:.4f})")
        else:
            result = {
                'original_text' : text,
                'final_text'    : text,
                'label'         : label,
                'confidence'    : confidence,
                'iterations'    : 1,
                'status'        : 'classified'
            }
            print(f"{label} (conf: {confidence:.4f})")

        registry.add(result, project_name=project_name)
        results.append(result)

    print(f"\n{'='*50}")
    print(f"Total processed : {len(results)}")
    print(f"Classified      : {len(results) - len(flagged)}")
    print(f"Flagged         : {len(flagged)}")

    if flagged:
        print(f"\nFlagged for manual review:")
        for r in flagged:
            print(f"  - {r}")

    return results

# ── Test batch processing ─────────────────────────────────────────────────
sample_requirements = [
    "The system shall allow users to reset their password via email.",
    "The system shall process transactions within 500 milliseconds.",
    "The system should be good and work well.",
    "Users should be able to do stuff easily.",
    "The system shall support role-based access control.",
    "The system shall maintain 99.9% uptime.",
    "It must handle everything properly.",
]

batch_results = process_requirements_batch(
    sample_requirements,
    project_name="Demo Project"
)

print("\nRegistry updated")
registry.summary(project_name="Demo Project")

Processing 7 requirements...

[1/7] The system shall allow users to reset their password via ema...
FR (conf: 0.9996)
[2/7] The system shall process transactions within 500 millisecond...
NFR_Performance (conf: 0.9983)
[3/7] The system should be good and work well....
Flagged as ambiguous (conf: 0.9990)
[4/7] Users should be able to do stuff easily....
Flagged as ambiguous (conf: 0.9990)
[5/7] The system shall support role-based access control....
Flagged as ambiguous (conf: 0.5980)
[6/7] The system shall maintain 99.9% uptime....
NFR_Reliability (conf: 0.6972)
[7/7] It must handle everything properly....
Flagged as ambiguous (conf: 0.9990)

Total processed : 7
Classified      : 3
Flagged         : 4

Flagged for manual review:
  - The system should be good and work well.
  - Users should be able to do stuff easily.
  - The system shall support role-based access control.
  - It must handle everything properly.

Registry updated
Total requirements : 44

By label:
label
Requires Manual R